# Домашнее задание № 2. Мешок слов

Для запуска нужны Python 3, pandas, NumPy, scikit-learn, pyarrow и razdel. При первом запуске скачиваются данные. На Windows код автоматически скачивает MyStem 3.1. На Linux/macOS можно указать путь к MyStem в MYSTEM_BIN. Для автоматического поиска через Python нужен pymystem3.

## Задание 1 (3 балла)

У векторайзеров в sklearn есть встроенная токенизация на регулярных выражениях. Найдите способо заменить её на кастомную токенизацию

Обучите векторайзер с дефолтной токенизацией, с токенизацией razdel.tokenize и с токенизацией+лемматизацией из mystem. Обучите классификатор (любой) с каждым из векторизаторов. Сравните метрики и выберите победителя. 

(в вашей тетрадке должен быть код обучения и все метрики; если вы сдаете в .py файлах то сохраните полученные метрики в отдельном файле или в комментариях)

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import json
import os
import re
import shutil
import subprocess
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from razdel import tokenize
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

SEED = 42
def dataset(name, url):
    path = Path(name)
    if not path.exists():
        urlretrieve(url, path)
    return path

def metrics(y, pred):
    return dict(accuracy=accuracy_score(y, pred), precision=precision_score(y, pred),
                recall=recall_score(y, pred), F1=f1_score(y, pred))

def split_rows(frame, target, size):
    return train_test_split(frame, test_size=size, stratify=frame[target], random_state=SEED)

In [2]:
ru_url = 'https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/notebooks/bow/labeled.csv.zip'
with ZipFile(dataset('labeled.csv.zip', ru_url)) as archive:
    ru = pd.read_csv(archive.open('labeled.csv')).dropna().copy()
ru['comment'] = ru.comment.str.lower().str.split().str.join(' ')
ru = ru.drop_duplicates('comment').reset_index(drop=True)
ru_train, ru_test = split_rows(ru, 'toxic', 0.2)
print(f'Обучение: {len(ru_train)}, тест: {len(ru_test)}')

Обучение: 11324, тест: 2831


У CountVectorizer и TfidfVectorizer параметр tokenizer принимает функцию, которая возвращает список токенов. При этом token_pattern отключается. Для MyStem заранее разбираю документы с учётом контекста и передаю готовые токены по тексту. Все три модели обучаются на одинаковых строках с одинаковыми настройками TF-IDF и логистической регрессии.

In [3]:
def razdel_words(text):
    return [token.text for token in tokenize(text)]

# Один вызов MyStem для корпуса. Каждая строка — отдельный документ.
binary = os.getenv('MYSTEM_BIN') or shutil.which('mystem')
if binary is None:
    if os.name == 'nt':
        mystem_url = 'https://download.cdn.yandex.net/mystem/mystem-3.1-win-64bit.zip'
        binary = Path('.mystem/mystem.exe').resolve()
        if not binary.exists():
            with ZipFile(dataset('mystem.zip', mystem_url)) as archive:
                archive.extract('mystem.exe', '.mystem')
    else:
        from pymystem3 import Mystem
        binary = Mystem()._mystem_bin
separator = 'zzdocumentboundaryzz'
assert not ru.comment.str.contains(separator).any()
with TemporaryDirectory() as folder:
    src, dst = (Path(folder) / name for name in ['input.txt', 'analysis.jsonl'])
    src.write_bytes((('\n' + separator + '\n').join(ru.comment) + '\n').encode('utf-8'))
    subprocess.run([binary, '--format', 'json', '-c', '-d', src, dst], check=True)
    analysis = dst.read_text(encoding='utf-8')
documents = [[]]
for line in analysis.split('\n'):
    if not line.strip():
        continue
    for t in json.loads(line):
        if t['text'] == separator:
            documents.append([])
        elif t['text'].strip():
            documents[-1].append(t['analysis'][0]['lex'] if t.get('analysis') else t['text'])
assert len(documents) == len(ru), (len(documents), len(ru), analysis[-200:])
lemmas = dict(zip(ru.comment, documents))

ru_scores = {}
for name, tokenizer in [('sklearn', None), ('razdel', razdel_words), ('MyStem', lemmas.__getitem__)]:
    options = {} if tokenizer is None else dict(tokenizer=tokenizer, token_pattern=None)
    model = make_pipeline(TfidfVectorizer(**options), LogisticRegression(C=4, max_iter=500))
    model.fit(ru_train.comment, ru_train.toxic)
    ru_scores[name] = metrics(ru_test.toxic, model.predict(ru_test.comment))
ru_table = pd.DataFrame(ru_scores).T
print(ru_table.round(4).to_string())
print('Победитель по F1 токсичного класса:', ru_table.F1.idxmax())

         accuracy  precision  recall      F1
sklearn    0.8509     0.8688  0.6519  0.7449
razdel     0.8481     0.8601  0.6508  0.7410
MyStem     0.8827     0.8846  0.7460  0.8094
Победитель по F1 токсичного класса: MyStem


## Задание 2 (3 балла)

Обучите 2 любых разных классификатора (используя алгоритмы из семинара). Используйте eng сабсет из `mteb/multi-hatecheck` в качестве датасета. 

Предскажите токсичность (`is_hateful`) для текстов из тестовой выборки (используйте одну и ту же выборку для обоих классификаторов) и найдите 10 самых токсичных для каждого из классификаторов (по вероятности). Сравните получаемые тексты - какие тексты совпадают, какие отличаются, правда ли тексты токсичные?

Требования к моделям:   
а) один классификатор должен использовать CountVectorizer, другой TfidfVectorizer  
б) у векторазера должны быть вручную заданы как минимум 5 параметров (можно ставить разные параметры tfidfvectorizer и countvectorizer)  
в) у классификатора должно быть задано вручную как минимум 2 параметра (по возможности)  
г)  f1 мера каждого из классификаторов должна быть минимум 0.60 

*random_seed, encoding, decode_error не считаются за параметры

В английском корпусе опубликована одна часть test, поэтому делю её самостоятельно. Удаляю точные повторы и сохраняю долю классов. Похожие варианты одного шаблона могут попасть в обе части, поэтому оценки характеризуют распознавание примеров внутри этого корпуса.

In [4]:
eng_url = 'https://huggingface.co/datasets/mteb/multi-hatecheck/resolve/main/test/eng.jsonl.gz'
eng = pd.read_json(dataset('eng.jsonl.gz', eng_url), lines=True)
eng['text'] = eng.text.str.strip()
eng = eng.drop_duplicates('text').copy()
eng['y'] = eng.is_hateful.map({'hateful': 1, 'non-hateful': 0})
assert eng.y.notna().all()
train, test = split_rows(eng, 'y', 0.2)
params = dict(lowercase=True, ngram_range=(1, 2), min_df=1, max_df=0.98, max_features=30000)
models = {
    'Count + NB': make_pipeline(CountVectorizer(binary=True, **params),
                                MultinomialNB(alpha=0.3, fit_prior=False)),
    'TF-IDF + LR': make_pipeline(TfidfVectorizer(sublinear_tf=True, **params),
                                 LogisticRegression(C=8, max_iter=600)),
}
print('Число текстов в обучении и тесте:', len(train), len(test))
rankings, scores = {}, {}
for name, model in models.items():
    model.fit(train.text, train.y)
    scores[name] = metrics(test.y, model.predict(test.text))
    ranked = test.assign(p=model.predict_proba(test.text)[:, 1]).sort_values(
        ['p', 'text'], ascending=[False, True]).head(10)
    rankings[name] = ranked
    print('\n', name)
    print(ranked[['text', 'p', 'is_hateful']].to_string(index=False))
print(pd.DataFrame(scores).T.round(4).to_string())
a, b = (set(r.text) for r in rankings.values())
for label, texts in [('Общие', a & b), ('Только NB', a - b), ('Только LR', b - a)]:
    print(label, len(texts), *sorted(texts), sep='\n')

Число текстов в обучении и тесте: 2982 746

 Count + NB
                                                                                   text   p  is_hateful
      A Muslim must have committed that crime. Only they can be that violent and cruel. 1.0     hateful
  A gay person must have committed that crime. Only they can be that violent and cruel. 1.0     hateful
       A woman must have committed that crime. Only they can be that violent and cruel. 1.0     hateful
  An immigrant must have committed that crime. Only they can be that violent and cruel. 1.0     hateful
                     As far as I am concerned, black people can all go fuck themselves. 1.0     hateful
                     As far as I am concerned, trans people can all go fuck themselves. 1.0     hateful
                     Centrists are without a doubt the stupidest people on this planet. 1.0 non-hateful
Gay people can be kind to you at times. Don't forget they're still just worthless scum. 1.0     hateful
       I

Две десятки не пересекаются. У NB четыре примера повторяют один шаблон: члену группы приписывают преступление на основании его происхождения, пола, религии или ориентации. Вероятности нескольких текстов округляются до 1, поэтому различия внутри этой части рейтинга малоинформативны. Девять из десяти имеют метку hateful. Исключение — реплика о центристах: она оскорбительна, но направлена против сторонников политического течения. Метка is_hateful здесь уже бытового понятия токсичности. У логистической регрессии все десять примеров размечены как hateful. В её десятке много коротких оскорблений, в том числе с намеренно изменённым написанием.

## Задание 3 (2 балла - 0,5 балл за каждый классификатор)

Для классификаторов Logistic Regression, Decision Trees, Naive Bayes, RandomForest найдите способ извлечь важность признаков для предсказания токсичного класса (любой из датасетов). Сопоставьте полученные числа со словами (или нграммами) в словаре и найдите топ - 5 "токсичных" слов для каждого из классификаторов. 

Важное требование: в топе не должно быть стоп-слов. Для этого вам нужно будет правильным образом настроить векторизацию. 
Также как и в предыдущем задании у классификаторов должно быть задано вручную как минимум 2 параметра (по возможности, f1 мера каждого из классификаторов должна быть минимум 0.6)

Словарь состоит из отдельных слов после удаления английских стоп-слов. У логистической регрессии беру коэффициент для класса 1, у наивного Байеса — логарифм отношения вероятностей слова в двух классах. Для дерева использую бинарные признаки: правая ветвь означает присутствие слова. Суммирую взвешенное уменьшение неоднородности со знаком изменения доли токсичного класса между ветвями; для леса усредняю результат по деревьям.

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop_words = sorted(ENGLISH_STOP_WORDS | {'just', 'don', 'didn', 'doesn', 'isn', 'aren', 've'})
vec = CountVectorizer(binary=True, stop_words=stop_words, min_df=2)
X = vec.fit_transform(train.text)
Xt = vec.transform(test.text)
words = vec.get_feature_names_out()
classifiers = {
    'LR': LogisticRegression(C=4, max_iter=500),
    'Дерево': DecisionTreeClassifier(max_depth=45, min_samples_leaf=2, random_state=SEED),
    'NB': MultinomialNB(alpha=0.4, fit_prior=False),
    'Лес': RandomForestClassifier(n_estimators=160, min_samples_leaf=1, max_features='sqrt',
                                  random_state=SEED, n_jobs=1),
}

def signed_tree_importance(tree):
    t = tree.tree_
    nodes = np.flatnonzero(t.feature >= 0)
    left, right = t.children_left[nodes], t.children_right[nodes]
    n = t.weighted_n_node_samples
    pos = t.value[:, 0, 1] / t.value[:, 0].sum(axis=1)
    gain = n[nodes]*t.impurity[nodes] - n[left]*t.impurity[left] - n[right]*t.impurity[right]
    return np.bincount(t.feature[nodes], weights=gain*np.sign(pos[right]-pos[left])/n[0],
                       minlength=len(words))

for name, clf in classifiers.items():
    clf.fit(X, train.y)
    if name == 'LR':
        weights = clf.coef_[0]
    elif name == 'NB':
        weights = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    elif name == 'Дерево':
        weights = signed_tree_importance(clf)
    else:
        weights = np.mean([signed_tree_importance(tree) for tree in clf.estimators_], axis=0)
    best = np.argsort(-weights, kind='stable')[:5]
    print(f'{name}, F1 = {f1_score(test.y, clf.predict(Xt)):.4f}')
    print(', '.join(f'{words[i]} ({weights[i]:.3f})' for i in best))

LR, F1 = 0.9538
head (4.711), immigrants (3.703), muslims (3.622), country (3.604), women (3.598)
Дерево, F1 = 0.8830
people (0.007), muslims (0.007), typical (0.006), immigrants (0.005), lives (0.005)
NB, F1 = 0.8824
planet (3.233), bunch (3.015), dirt (3.015), natural (3.015), care (2.813)
Лес, F1 = 0.9191
people (0.007), immigrants (0.006), muslims (0.006), women (0.005), country (0.003)


Веса относятся к тому, как составлен корпус: например, упоминание группы может помогать распознавать повторяющийся шаблон. Само название группы не выражает ненависти. Значения весов сопоставимы внутри одной модели, поскольку методы важности используют разные шкалы.

# Задание 4* (2 балла)

Составьте ансамблевый классификатор вручную. Используя один из парных датасетов в `glue` (например, `mnli`) обучите как минимум 10 разных классификаторов (так как количество алгоритмов меньше 10, используйте разные комбинации параметров в векторайзерах и в классификаторах, например, tfidf + logreg и countvectorizer+logreg). Чем сильнее каждый классификатор отличается от другого, тем лучше. 

Вместо стандартного разбиения на train и test, разбейте выборку на 3 части: train, dev и test. Используйте train для обучения 10 классификаторов. Сделайте предсказания всеми классификаторами для dev и test датасетов. Используя объединенные предсказания всех классификаторов на dev части как признаки, обучите еще один общий классификатор. Сделайте предсказания общим классификатором на test части (опять же используйте предсказания 10 классификаторов как признаки) и рассчитайте качество итоговой классификации. 

Также отдельно оцените качество на test части для каждого из 10 классификаторов. Сравните с общей оценкой. Превосходит ли общий классификатор в качестве самый лучший из отдельных классификаторов? 

Использую размеченную обучающую часть GLUE/MRPC и делю её в отношении 60/20/20. Класс 1 означает, что предложения являются парафразами. Каждая пара кодируется симметрично: отдельно обозначаю общие и несовпадающие слова, а долю пересечения словарей кодирую по интервалам.

In [6]:
mrpc_url = 'https://huggingface.co/datasets/nyu-mll/glue/resolve/main/mrpc/train-00000-of-00001.parquet'
pairs = pd.read_parquet(dataset('mrpc.parquet', mrpc_url))
def pair_document(row):
    a, b = (set(re.findall(r'\b\w+\b', text.lower())) for text in (row.sentence1, row.sentence2))
    overlap = len(a & b) / max(1, len(a | b))
    return ' '.join([*(f'shared_{w}' for w in sorted(a & b)),
                     *(f'diff_{w}' for w in sorted(a ^ b)), f'overlap_{int(10*overlap)}'])

pairs['doc'] = pairs.apply(pair_document, axis=1)
pairs = pairs.drop_duplicates('doc')
base, held = split_rows(pairs, 'label', 0.4)
dev, final = split_rows(held, 'label', 0.5)
print('Размеры train/dev/test:', len(base), len(dev), len(final))

Размеры train/dev/test: 2190 730 730


Каждая базовая модель передаёт на следующий уровень одну вероятность парафраза. Десять столбцов на dev служат обучающими признаками метамодели. На test порядок столбцов тот же. Основная метрика сравнения — F1 для парафразов.

In [7]:
bow = CountVectorizer(min_df=2, binary=True)
tf = TfidfVectorizer(min_df=2, sublinear_tf=True)
char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=4, max_features=15000)
specs = [
    ('Count LR', bow, LogisticRegression(C=0.2, max_iter=500)),
    ('TF-IDF LR', tf, LogisticRegression(C=5, max_iter=500)),
    ('Char LR', char, LogisticRegression(C=4, max_iter=500)),
    ('Count NB', bow, MultinomialNB(alpha=1)),
    ('TF-IDF NB', tf, MultinomialNB(alpha=0.1)),
    ('Char ComplementNB', char, ComplementNB(alpha=0.5)),
    ('Count Tree', bow, DecisionTreeClassifier(max_depth=10, random_state=SEED)),
    ('TF-IDF Tree', tf, DecisionTreeClassifier(max_depth=20, min_samples_leaf=6, random_state=SEED)),
    ('Count Forest', bow, RandomForestClassifier(n_estimators=100, random_state=SEED)),
    ('TF-IDF ExtraTrees', tf, ExtraTreesClassifier(n_estimators=100, random_state=SEED)),
]
dev_predictions, test_predictions, comparison = {}, {}, {}
for name, vectorizer, classifier in specs:
    model = make_pipeline(clone(vectorizer), classifier).fit(base.doc, base.label)
    dev_predictions[name] = model.predict_proba(dev.doc)[:, 1]
    test_predictions[name] = model.predict_proba(final.doc)[:, 1]
    comparison[name] = metrics(final.label, model.predict(final.doc))

meta = LogisticRegression(C=1, max_iter=500).fit(pd.DataFrame(dev_predictions), dev.label)
ensemble = metrics(final.label, meta.predict(pd.DataFrame(test_predictions)))
table = pd.DataFrame(comparison).T
best = table.F1.idxmax()
table.loc['Ансамбль'] = ensemble
print(table.round(4).to_string())
print(f'Лучший базовый по F1: {best}. Разница F1 (ансамбль минус лучший): {ensemble["F1"] - table.loc[best,"F1"]:+.4f}')

                   accuracy  precision  recall      F1
Count LR             0.7370     0.7636  0.8844  0.8195
TF-IDF LR            0.7301     0.7624  0.8722  0.8136
Char LR              0.7452     0.7717  0.8844  0.8242
Count NB             0.7164     0.7581  0.8519  0.8023
TF-IDF NB            0.6959     0.7240  0.8884  0.7978
Char ComplementNB    0.7096     0.7206  0.9310  0.8124
Count Tree           0.7178     0.7702  0.8296  0.7988
TF-IDF Tree          0.6356     0.7379  0.7140  0.7258
Count Forest         0.7260     0.7329  0.9351  0.8217
TF-IDF ExtraTrees    0.7356     0.7459  0.9229  0.8250
Ансамбль             0.7507     0.7733  0.8925  0.8286
Лучший базовый по F1: TF-IDF ExtraTrees. Разница F1 (ансамбль минус лучший): +0.0036


На отложенных 730 парах ансамбль получил F1 = 0,8286, лучшая базовая модель (ExtraTrees с TF-IDF) — 0,8250. Выигрыш — 0,0036. У ExtraTrees выше полнота (0,9229), а у ансамбля выше точность положительных предсказаний (0,7733 против 0,7459). Вывод о небольшом преимуществе относится к этому разбиению.